# B3 · Polynomial pipeline with Ridge and GridSearchCV

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Build a `Pipeline` (PolynomialFeatures, StandardScaler, Ridge), choose degree and regularization with
`GridSearchCV`, and measure what happens outside the range of the data with a degree that is too high.

## Dataset

`ds.energy_consumption()`: 160 days, outside temperature in degrees Celsius (from −5 to 35) and energy consumption in kWh.
The relationship is U-shaped: more is used in the cold (heating) and in the heat (air conditioning).

## Educational purpose

Use `Pipeline` and `GridSearchCV` with nested parameters (`step__parameter`), and understand that **a good test error
does not guarantee a good model when you extrapolate**.

## Task

1. Split into training and test (70/30, `random_state=0`).
2. Build `pipe = Pipeline([("poly", PolynomialFeatures(include_bias=False)), ("scaler", StandardScaler()), ("ridge", Ridge())])`.
3. With `GridSearchCV` search `poly__degree` from 1 to 10 and `ridge__alpha` in `np.logspace(-4, 2, 7)`, with `cv=KFold(5, shuffle=True, random_state=0)` and `scoring="neg_root_mean_squared_error"`. Save `search`, `chosen_degree`, `chosen_alpha`, `rmse_cv` (positive!) and `rmse_test` (only once, at the end).
4. Train two comparison models: `linear` (`LinearRegression`, degree 1) and `poly10` (degree 10 with `LinearRegression`, without regularization, with the scaler). Compute `rmse_test_linear` and `rmse_test_poly10`.
5. Predict at 45 degrees (`t45`, outside the training range) with the three models: `prediction_45_chosen`, `prediction_45_linear`, `prediction_45_poly10`. The value of the generating curve is `true_45`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

d = ds.energy_consumption()
X, y = d[["temperature"]], d["consumption"]
t45 = pd.DataFrame({"temperature": [45.0]})
true_45 = 50 + 0.16 * (45 - 17) ** 2               # the true curve, which the model does not know
print("true value at 45 degrees:", round(true_45, 1), "kWh")

## Your work

In [ ]:
X_tr, X_te, y_tr, y_te = ...
pipe = ...

In [ ]:
grid = {"poly__degree": ..., "ridge__alpha": ...}
search = GridSearchCV(pipe, grid, cv=..., scoring=...)
search.fit(X_tr, y_tr)

chosen_degree = ...
chosen_alpha = ...
rmse_cv = ...
rmse_test = ...
print("degree", chosen_degree, "| alpha", chosen_alpha, "| RMSE cv", round(rmse_cv, 2), "| RMSE test", round(rmse_test, 2))

In [ ]:
linear = ...
poly10 = ...
rmse_test_linear = ...
rmse_test_poly10 = ...

In [ ]:
prediction_45_chosen = ...
prediction_45_linear = ...
prediction_45_poly10 = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='To access the results: `search.best_params_["poly__degree"]`, `search.best_score_` (negative: the RMSE is `-search.best_score_`). The unregularized degree 10 polynomial is `Pipeline([("poly", PolynomialFeatures(10, include_bias=False)), ("scaler", StandardScaler()), ("lr", LinearRegression())])`.'):
    check(len(X_te) == 48, "70/30 split", "X_te should have 48 rows (30% of 160)")
    check(2 <= chosen_degree <= 5, "the search chooses a low degree (the true curve is quadratic)",
          "the chosen degree should be between 2 and 5: check the scoring and the grid")
    check(rmse_cv > 0, "rmse_cv is positive", "use the minus sign: best_score_ is a negative score")
    check(close(rmse_test, mean_squared_error(y_te, search.predict(X_te), squared=False)),
          "rmse_test is computed on the test set", "rmse_test must be the RMSE of search.predict(X_te)")
    check(rmse_test_linear > 2 * rmse_test, "the line is much worse than the chosen model (the trend is U-shaped)",
          "the linear model should have an RMSE at least double")
    check(rmse_test_poly10 < 1.3 * rmse_test,
          "the degree 10 polynomial has an almost equal test error: INSIDE the range you don't see the problem",
          "degree 10 with the scaler should have a comparable test error: check that you used the scaler")
    check(abs(prediction_45_chosen - true_45) < 20, "the chosen model extrapolates reasonably",
          "the prediction at 45 degrees of the chosen model should be close to the true value")
    check(abs(prediction_45_poly10) > 1000, "degree 10 explodes outside the range",
          "the degree 10 polynomial should give an absurd prediction (thousands of kWh) at 45 degrees")
done()

---
The commented solution is in `solutions/B_with_sklearn/B3_polynomial_ridge_gridsearch_pipeline.ipynb`. Open it after you pass the check, or after you have really tried.